## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 11: Autoencoder
Niveau: Fortgeschrittene
Aufgabe 11.1: Variational Autoencoder (VAE) – Theorie und Implementierung

Lernziel: VAE mit Reparametrization Trick implementieren
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, _), (X_test, _) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

LATENT_DIM = 2

class Sampling(keras.layers.Layer):
    """Reparametrization Trick: z = mu + eps * sigma."""
    def call(self, inputs):
        mu, log_var = inputs
        batch = tf.shape(mu)[0]
        dim   = tf.shape(mu)[1]
        eps = tf.random.normal(shape=(batch, dim))
        return mu + tf.exp(0.5 * log_var) * eps

# Encoder
encoder_inputs = keras.Input(shape=(784,))
x = keras.layers.Dense(256, activation='relu')(encoder_inputs)
x = keras.layers.Dense(128, activation='relu')(x)
mu      = keras.layers.Dense(LATENT_DIM, name='mu')(x)
log_var = keras.layers.Dense(LATENT_DIM, name='log_var')(x)
z = Sampling()([mu, log_var])
encoder = keras.Model(encoder_inputs, [mu, log_var, z], name='Encoder')

# Decoder
decoder_inputs = keras.Input(shape=(LATENT_DIM,))
x = keras.layers.Dense(128, activation='relu')(decoder_inputs)
x = keras.layers.Dense(256, activation='relu')(x)
decoder_outputs = keras.layers.Dense(784, activation='sigmoid')(x)
decoder = keras.Model(decoder_inputs, decoder_outputs, name='Decoder')

# VAE Klasse
class VAE(keras.Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def train_step(self, data):
        with tf.GradientTape() as tape:
            mu, log_var, z = self.encoder(data)
            rekonstruiert = self.decoder(z)
            
            # Rekonstruktionsverlust
            recon_loss = tf.reduce_mean(
                keras.losses.binary_crossentropy(data, rekonstruiert)
            ) * 784
            
            # KL-Divergenz-Verlust
            kl_loss = -0.5 * tf.reduce_mean(
                1 + log_var - tf.square(mu) - tf.exp(log_var)
            )
            
            total_loss = recon_loss + kl_loss
        
        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {'loss': total_loss, 'recon_loss': recon_loss, 'kl_loss': kl_loss}

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(1e-3))
history = vae.fit(X_train, epochs=20, batch_size=256, verbose=1)

# Latent Space Visualisierung
_, _, z_test = encoder.predict(X_test, verbose=0)
(_, y_test_orig), _ = keras.datasets.mnist.load_data()

plt.figure(figsize=(10, 8))
scatter = plt.scatter(z_test[:, 0], z_test[:, 1],
                       c=y_test_orig, cmap='tab10', alpha=0.5, s=5)
plt.colorbar(scatter, label='Ziffer')
plt.title('VAE Latent Space (2D)')
plt.xlabel('z₁')
plt.ylabel('z₂')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vae_latent_space.png', dpi=150)
plt.show()

# Neue Bilder generieren durch Sampling
print("\nGeneriere neue Bilder aus dem Latent Space...")
z_neu = np.random.randn(10, LATENT_DIM).astype('float32')
generiert = decoder.predict(z_neu, verbose=0)

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(generiert[i].reshape(28, 28), cmap='gray')
    ax.axis('off')
plt.suptitle('VAE Generierte Bilder (zufälliges Sampling)')
plt.tight_layout()
plt.savefig('vae_generiert.png', dpi=150)
plt.show()


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 11: Autoencoder
Niveau: Fortgeschrittene
Aufgabe 11.2: Anomalie-Detektion mit Autoencoder

Lernziel: Autoencoder zur Erkennung von Ausreißern nutzen
Datensatz: MNIST (Klasse 0 als normal, andere als Anomalie)
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import roc_auc_score, roc_curve

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

# Nur Klasse "0" zum Training (normal)
X_normal_tr = X_train[y_train == 0]
print(f"Normal Trainingsdaten (Ziffer 0): {len(X_normal_tr)}")

# Testdaten: 0 = normal, rest = Anomalie
X_test_all = X_test[:2000]
y_test_labels = (y_test[:2000] != 0).astype(int)  # 1 = Anomalie
print(f"Test: {len(X_test_all)} Bilder, davon {y_test_labels.sum()} Anomalien")

# Autoencoder trainieren (nur auf normalen Daten)
eingabe = keras.Input(shape=(784,))
x = keras.layers.Dense(256, activation='relu')(eingabe)
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dense(16, activation='relu')(x)
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dense(256, activation='relu')(x)
ausgabe = keras.layers.Dense(784, activation='sigmoid')(x)

ae = keras.Model(eingabe, ausgabe)
ae.compile(optimizer='adam', loss='mse')
ae.fit(X_normal_tr, X_normal_tr, epochs=20, batch_size=256, verbose=0)

# Rekonstruktionsfehler als Anomalie-Score
X_rekonstruiert = ae.predict(X_test_all, verbose=0)
anomalie_scores = np.mean((X_test_all - X_rekonstruiert)**2, axis=1)

# ROC-AUC
auc = roc_auc_score(y_test_labels, anomalie_scores)
fpr, tpr, schwellen = roc_curve(y_test_labels, anomalie_scores)
print(f"\nROC-AUC: {auc:.4f}")

# Optimale Schwelle (Youden Index)
opt_idx = np.argmax(tpr - fpr)
opt_schwelle = schwellen[opt_idx]
y_pred = (anomalie_scores > opt_schwelle).astype(int)
acc = np.mean(y_pred == y_test_labels)
print(f"Optimale Schwelle: {opt_schwelle:.4f}")
print(f"Accuracy: {acc*100:.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(fpr, tpr, color='#00E6FF', linewidth=2)
axes[0].plot([0,1],[0,1],'--', color='gray')
axes[0].scatter(fpr[opt_idx], tpr[opt_idx], color='red', s=100, zorder=5)
axes[0].set_title(f'ROC Kurve (AUC={auc:.3f})')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].grid(True, alpha=0.3)

axes[1].hist(anomalie_scores[y_test_labels==0], bins=50,
             alpha=0.6, label='Normal (0)', color='#00E6FF')
axes[1].hist(anomalie_scores[y_test_labels==1], bins=50,
             alpha=0.6, label='Anomalie', color='#FF6B6B')
axes[1].axvline(opt_schwelle, color='red', linestyle='--', label='Schwelle')
axes[1].set_title('Score-Verteilungen')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

schlechteste_normale = X_test_all[y_test_labels==0][np.argsort(
    anomalie_scores[y_test_labels==0])[-5:]]
beste_anomalien = X_test_all[y_test_labels==1][np.argsort(
    anomalie_scores[y_test_labels==1])[:5]]

for i in range(5):
    ax = fig.add_subplot(3, 5, 11+i)
    ax.imshow(schlechteste_normale[i].reshape(28,28), cmap='gray')
    ax.axis('off')

plt.suptitle('Anomalie-Detektion mit Autoencoder')
plt.tight_layout()
plt.savefig('anomalie_detektion_ae.png', dpi=150)
plt.show()


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 11: Autoencoder
Niveau: Fortgeschrittene
Aufgabe 11.3: Sparse Autoencoder

Lernziel: Sparsity-Regularisierung für interpretierbarere Repräsentationen
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

LATENT_DIM = 64
SPARSITY_LAMBDA = 1e-5

# Standard Autoencoder
def standard_ae():
    eingabe = keras.Input(shape=(784,))
    x = keras.layers.Dense(256, activation='relu')(eingabe)
    latent = keras.layers.Dense(LATENT_DIM, activation='relu')(x)
    x = keras.layers.Dense(256, activation='relu')(latent)
    ausgabe = keras.layers.Dense(784, activation='sigmoid')(x)
    return keras.Model(eingabe, ausgabe, name='StandardAE')

# Sparse Autoencoder (L1 auf Aktivierungen)
def sparse_ae():
    eingabe = keras.Input(shape=(784,))
    x = keras.layers.Dense(256, activation='relu')(eingabe)
    latent = keras.layers.Dense(LATENT_DIM, activation='sigmoid',
                                 activity_regularizer=keras.regularizers.l1(SPARSITY_LAMBDA))(x)
    x = keras.layers.Dense(256, activation='relu')(latent)
    ausgabe = keras.layers.Dense(784, activation='sigmoid')(x)
    return keras.Model(eingabe, ausgabe, name='SparseAE')

modelle = {'Standard AE': standard_ae(), 'Sparse AE': sparse_ae()}
histories = {}
encoder_modelle = {}

for name, ae in modelle.items():
    ae.compile(optimizer='adam', loss='binary_crossentropy')
    h = ae.fit(X_train, X_train, epochs=15, batch_size=256, verbose=0)
    histories[name] = h
    # Encoder extrahieren
    encoder_modelle[name] = keras.Model(ae.input, ae.layers[2].output)
    
    # Sparsity messen
    repr_ = encoder_modelle[name].predict(X_test[:1000], verbose=0)
    sparsity = np.mean(repr_ < 0.1)
    mse = np.mean((X_test[:100] - ae.predict(X_test[:100], verbose=0))**2)
    print(f"{name}: MSE={mse:.4f}, Sparsity={sparsity*100:.1f}%")

# Latente Aktivierungen visualisieren
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, (name, enc) in enumerate(encoder_modelle.items()):
    repr_ = enc.predict(X_test[:500], verbose=0)
    aktiviert = (repr_ > 0.1).mean(axis=0)
    axes[i].bar(range(LATENT_DIM), aktiviert, color='#00E6FF')
    axes[i].set_title(f'{name} – Aktivierungsrate pro Neuron')
    axes[i].set_xlabel('Latente Einheit')
    axes[i].set_ylabel('Aktivierungsrate')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Standard vs. Sparse Autoencoder')
plt.tight_layout()
plt.savefig('sparse_autoencoder.png', dpi=150)
plt.show()
